In [4]:
import plotly.express as px
import pandas as pd
import plotly.graph_objects as go
from dash import Dash, dcc
import dash_ag_grid as dag

# Download CSV sheet at: https://drive.google.com/file/d/1CXxOQA2uBso64VEyvQ3L76AZLxyUlQDB/view?usp=sharing
df = pd.read_csv("OECD-wellbeing-OECD-wellbeing.csv")

# Group the dataset
df_grouped = df.groupby(['Measure', 'Education', 'Country', 'Year'])['OBS_VALUE'].sum()
df_grouped = df_grouped.reset_index()

df_grouped

,Measure,Education,Country,Year,OBS_VALUE
0,Access to green space,Total,Austria,2012,77.446842
1,Access to green space,Total,Austria,2018,77.252527
2,Access to green space,Total,Belgium,2012,57.353296
3,Access to green space,Total,Belgium,2018,57.652810
4,Access to green space,Total,Bulgaria,2012,62.758527
...,...,...,...,...,...
46045,"Youth not in employment, education or training",Total,United States,2019,34.734230
46046,"Youth not in employment, education or training",Total,United States,2020,34.760306
46047,"Youth not in employment, education or training",Total,United States,2021,42.277166
46048,"Youth not in employment, education or training",Total,United States,2022,38.902519


In [ ]:


# focus on a specific measure, year, and education level
df_grouped = df_grouped[df_grouped['Measure'] == 'Feeling lonely']
df_grouped = df_grouped[df_grouped['Year'] == 2022]
df_grouped = df_grouped[df_grouped['Education'].isin(['Primary education', 'Secondary education'])]

# Pivot the table to get Primary and Secondary values side-by-side
df_pivot = df_grouped.pivot(index='Country', columns='Education', values='OBS_VALUE').reset_index()

df_pivot.columns.name = None # Remove the name of the index colum

df_pivot = df_pivot.rename(columns={
    'Primary education': 'Primary',
    'Secondary education': 'Secondary'
})


# Sort countries
df_pivot = df_pivot.sort_values(by='Secondary', ascending=True)

# Get the sorted list of countries for the y-axis
countries_sorted = df_pivot['Country'].tolist()

# Prepare Data for Plotly Traces
line_x = []
line_y = []
primary_vals = []
secondary_vals = []

for country in countries_sorted:
    row = df_pivot[df_pivot['Country'] == country]
    primary_val = row['Primary'].iloc[0]
    secondary_val = row['Secondary'].iloc[0]

    primary_vals.append(primary_val)
    secondary_vals.append(secondary_val)

    # For the connecting line segment
    line_x.extend([primary_val, secondary_val, None]) # Add None to break the line
    line_y.extend([country, country, None])


fig = go.Figure()
fig.add_trace(go.Scatter(
    x=line_x,
    y=line_y,
    mode='lines',
    showlegend=False,
    line=dict(color='grey', width=1),
))

# Add markers for Primary Education
fig.add_trace(go.Scatter(
    x=primary_vals,
    y=countries_sorted,
    mode='markers',
    name='Primary Education', # Legend entry
    marker=dict(color='skyblue', size=10),
    hovertemplate =
        '<b>%{y}</b><br>' +
        'Primary Education: %{x:.2f}%' +
        '<extra></extra>'
))

# Add markers for Secondary Education
fig.add_trace(go.Scatter(
    x=secondary_vals,
    y=countries_sorted,
    mode='markers',
    name='Secondary Education', # Legend entry
    marker=dict(color='royalblue', size=10),
    hovertemplate =
        '<b>%{y}</b><br>' +
        'Secondary Education: %{x:.2f}%' +
        '<extra></extra>'
))

fig.update_layout(
    title=dict(text="Feeling Lonely by Education Level (2022)", x=0.5),
    xaxis_title="Percentage Feeling Lonely (%)",
    yaxis_title="Country",
    height=900,
    yaxis=dict(tickmode='array', tickvals=countries_sorted, ticktext=countries_sorted), # Ensure all country labels are shown
    legend_title_text='Education Level',
    legend=dict(
        orientation="h", # Horizontal legend
        yanchor="bottom",
        y=1.02, # Position above plot
        xanchor="right",
        x=1
    ),
    margin=dict(l=100) # Add left margin for country names
)


grid = dag.AgGrid(
    rowData=df.to_dict("records"),
    columnDefs=[{"field": i, 'filter': True, 'sortable': True} for i in df.columns],
    dashGridOptions={"pagination": True},
    columnSize="sizeToFit"
)

app = Dash()
app.layout = [
    grid,
    dcc.Graph(figure=fig)
]


if __name__ == "__main__":
    app.run(debug=False,port=8050)